# Comparativa de proveedores de IA

Benchmark práctico entre Claude (Anthropic), GPT (OpenAI), Gemini (Google) y Mistral:
calidad de respuesta, latencia, coste y casos de uso óptimos.

In [ ]:
import os, time, json
import anthropic
from openai import OpenAI

cliente_anthropic = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
cliente_openai = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print('Clientes inicializados')

## 1. Adaptador unificado para múltiples proveedores

In [ ]:
from dataclasses import dataclass

@dataclass
class ResultadoLLM:
    proveedor: str
    modelo: str
    respuesta: str
    tokens_entrada: int
    tokens_salida: int
    latencia_ms: float
    coste_usd: float

PRECIOS = {
    'claude-haiku-4-5-20251001':  {'entrada': 0.80,  'salida': 4.00},
    'claude-sonnet-4-6':          {'entrada': 3.00,  'salida': 15.00},
    'gpt-4o-mini':                {'entrada': 0.15,  'salida': 0.60},
    'gpt-4o':                     {'entrada': 2.50,  'salida': 10.00},
    'gemini-1.5-flash':           {'entrada': 0.075, 'salida': 0.30},
    'mistral-small-latest':       {'entrada': 0.10,  'salida': 0.30},
}

def calcular_coste(modelo: str, tokens_entrada: int, tokens_salida: int) -> float:
    p = PRECIOS.get(modelo, {'entrada': 1.0, 'salida': 3.0})
    return (tokens_entrada * p['entrada'] + tokens_salida * p['salida']) / 1_000_000

def llamar_claude(pregunta: str, modelo: str = 'claude-haiku-4-5-20251001') -> ResultadoLLM:
    inicio = time.time()
    resp = cliente_anthropic.messages.create(
        model=modelo, max_tokens=512,
        messages=[{'role': 'user', 'content': pregunta}]
    )
    latencia = (time.time() - inicio) * 1000
    ti, ts = resp.usage.input_tokens, resp.usage.output_tokens
    return ResultadoLLM('Anthropic', modelo, resp.content[0].text, ti, ts,
                        latencia, calcular_coste(modelo, ti, ts))

def llamar_openai(pregunta: str, modelo: str = 'gpt-4o-mini') -> ResultadoLLM:
    inicio = time.time()
    resp = cliente_openai.chat.completions.create(
        model=modelo, max_tokens=512,
        messages=[{'role': 'user', 'content': pregunta}]
    )
    latencia = (time.time() - inicio) * 1000
    ti = resp.usage.prompt_tokens
    ts = resp.usage.completion_tokens
    return ResultadoLLM('OpenAI', modelo, resp.choices[0].message.content, ti, ts,
                        latencia, calcular_coste(modelo, ti, ts))

print('Adaptadores listos')

## 2. Benchmark con tareas representativas

In [ ]:
TAREAS_BENCHMARK = [
    {
        'nombre': 'Razonamiento matemático',
        'pregunta': 'Si un tren sale a las 9:00 a 120 km/h y otro sale a las 10:30 a 180 km/h en dirección contraria, ¿a qué distancia de la primera ciudad se cruzan si están a 630 km?',
    },
    {
        'nombre': 'Resumen de texto',
        'pregunta': 'Resume en 3 puntos clave: Los Large Language Models (LLMs) han transformado el procesamiento del lenguaje natural. Utilizan arquitecturas transformer con millones de parámetros entrenados en enormes corpus de texto. Su capacidad emergente para razonar, generar código y mantener conversaciones coherentes ha democratizado el acceso a la IA. Sin embargo, presentan limitaciones como alucinaciones, sesgos heredados del corpus de entrenamiento y alto coste computacional.',
    },
    {
        'nombre': 'Generación de código',
        'pregunta': 'Escribe una función Python que calcule el número de Fibonacci de forma eficiente con memoización.',
    },
]

resultados: list[ResultadoLLM] = []

for tarea in TAREAS_BENCHMARK:
    print(f'\n📊 Tarea: {tarea["nombre"]}')
    for fn, modelo in [(llamar_claude, 'claude-haiku-4-5-20251001'), (llamar_openai, 'gpt-4o-mini')]:
        try:
            r = fn(tarea['pregunta'], modelo)
            resultados.append(r)
            print(f'  {r.proveedor} ({r.modelo}): {r.latencia_ms:.0f}ms, ${r.coste_usd:.6f}')
        except Exception as e:
            print(f'  Error: {e}')

print(f'\nTotal llamadas: {len(resultados)}')

## 3. Análisis de resultados

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

if resultados:
    df = pd.DataFrame([
        {
            'proveedor': r.proveedor,
            'modelo': r.modelo,
            'latencia_ms': r.latencia_ms,
            'coste_usd': r.coste_usd,
            'tokens_salida': r.tokens_salida,
        } for r in resultados
    ])

    print('=== Resumen por proveedor ===')
    print(df.groupby('proveedor').agg(
        latencia_media=('latencia_ms', 'mean'),
        coste_total=('coste_usd', 'sum'),
        tokens_salida_medio=('tokens_salida', 'mean'),
    ).round(4))
else:
    # Datos de ejemplo si no hay claves de API
    datos_ejemplo = {
        'proveedor': ['Anthropic', 'Anthropic', 'OpenAI', 'OpenAI', 'Google', 'Mistral'],
        'modelo': ['Haiku', 'Sonnet', 'GPT-4o-mini', 'GPT-4o', 'Gemini Flash', 'Mistral Small'],
        'latencia_ms': [450, 1200, 380, 2100, 290, 410],
        'coste_por_1M_tokens': [2.4, 9.0, 0.375, 6.25, 0.1875, 0.2],
        'calidad_razonamiento': [7, 9, 8, 9, 7, 7],
        'calidad_codigo': [8, 9, 8, 9, 7, 8],
        'calidad_multilenguaje': [8, 9, 7, 8, 8, 9],
    }
    df = pd.DataFrame(datos_ejemplo)
    print('Usando datos de referencia (sin claves de API):')
    print(df.to_string(index=False))

## 4. Calculadora de TCO (Total Cost of Ownership)

In [ ]:
def calcular_tco_mensual(
    llamadas_dia: int,
    tokens_entrada_media: int,
    tokens_salida_media: int,
) -> dict:
    llamadas_mes = llamadas_dia * 30
    tco = {}
    for modelo, precios in PRECIOS.items():
        coste_por_llamada = (
            tokens_entrada_media * precios['entrada'] +
            tokens_salida_media * precios['salida']
        ) / 1_000_000
        tco[modelo] = {
            'coste_por_llamada': coste_por_llamada,
            'coste_mensual': coste_por_llamada * llamadas_mes,
            'coste_anual': coste_por_llamada * llamadas_mes * 12,
        }
    return tco

# Caso de uso: chatbot de soporte con 500 conversaciones/día
tco = calcular_tco_mensual(
    llamadas_dia=500,
    tokens_entrada_media=800,   # contexto + historial
    tokens_salida_media=200,    # respuesta
)

print('TCO para 500 llamadas/día (800 tokens entrada, 200 salida):')
print(f'{"Modelo":<35} {"$/llamada":>12} {"$/mes":>10} {"$/año":>12}')
print('-' * 72)
for modelo, costes in sorted(tco.items(), key=lambda x: x[1]['coste_mensual']):
    print(f'{modelo:<35} ${costes["coste_por_llamada"]:>10.5f} ${costes["coste_mensual"]:>8.2f} ${costes["coste_anual"]:>10.2f}')

## 5. Guía de selección de proveedor

In [ ]:
from IPython.display import display, HTML

guia = """
<table style='border-collapse:collapse;width:100%;font-family:sans-serif;font-size:13px'>
<tr style='background:#1a1a2e;color:white'>
  <th style='padding:10px'>Caso de uso</th>
  <th style='padding:10px'>Recomendación</th>
  <th style='padding:10px'>Razón</th>
</tr>
<tr style='background:#f8f9fa'>
  <td style='padding:8px'>Clasificación / tareas simples</td>
  <td style='padding:8px;color:#388E3C;font-weight:bold'>Claude Haiku / GPT-4o-mini / Gemini Flash</td>
  <td style='padding:8px'>Máxima relación calidad/precio</td>
</tr>
<tr>
  <td style='padding:8px'>Análisis complejo / razonamiento</td>
  <td style='padding:8px;color:#1565C0;font-weight:bold'>Claude Sonnet / GPT-4o</td>
  <td style='padding:8px'>Mejor capacidad de razonamiento multi-paso</td>
</tr>
<tr style='background:#f8f9fa'>
  <td style='padding:8px'>Código y depuración</td>
  <td style='padding:8px;color:#1565C0;font-weight:bold'>Claude Sonnet / GPT-4o</td>
  <td style='padding:8px'>Mejor comprensión de contexto de código largo</td>
</tr>
<tr>
  <td style='padding:8px'>Multilingüe / europeo</td>
  <td style='padding:8px;color:#6A1B9A;font-weight:bold'>Mistral Large / Gemini</td>
  <td style='padding:8px'>Entrenados con más corpus europeo; GDPR-friendly</td>
</tr>
<tr style='background:#f8f9fa'>
  <td style='padding:8px'>Ventana de contexto enorme</td>
  <td style='padding:8px;color:#E65100;font-weight:bold'>Gemini 1.5 Pro (1M tokens)</td>
  <td style='padding:8px'>Mayor ventana disponible comercialmente</td>
</tr>
<tr>
  <td style='padding:8px'>Multimodalidad avanzada</td>
  <td style='padding:8px;color:#1565C0;font-weight:bold'>GPT-4o / Claude Sonnet</td>
  <td style='padding:8px'>Mejor comprensión de imagen+texto combinados</td>
</tr>
</table>
"""

display(HTML(guia))